# Automated Multilingual CEFR Classification — Kaggle Runner

Runs **Exp 0 – Exp 4** (baselines → CORAL → LLaMA+LoRA) on Kaggle GPU (H100 / T4).

| Experiment | Type | GPU? |
|---|---|---|
| Exp 0 – Majority | baseline | — |
| Exp 1 – TF-IDF+LR | baseline | — |
| Exp 7 – TF-IDF+LinearSVC | baseline | — |
| Exp 9 – Word TF-IDF+LR | ablation | — |
| Exp 2 – XLM-R fine-tuned | transformer | ✓ |
| Exp 3 – CORAL ordinal | transformer | ✓ |
| Exp 8 – Zero-shot XLM-R | zero-shot | ✓ |
| Exp 4 – LLaMA-3.2+LoRA (raw + constrained) | LLM | ✓ HF_TOKEN |

> **HF_TOKEN** — add your Hugging Face token as a Kaggle Secret named `HF_TOKEN`  
> (required only for Exp 4; must have accepted the LLaMA-3.2 license on HF Hub)

---

## 1. Environment Setup

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", result.stdout.strip() or "No GPU detected")

import torch
print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  "
      f"device = {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Install / upgrade packages not bundled with Kaggle's base image
!pip install -q \
    "datasets>=2.14.0" \
    "transformers>=4.44.0" \
    "accelerate>=0.30.0" \
    "peft>=0.11.0" \
    "bitsandbytes>=0.43.0" \
    "evaluate>=0.4.0" \
    "scikit-learn>=1.3.0" \
    "langdetect>=1.0.9"

print("\n✓ Dependencies installed")

In [ ]:
import os, sys

REPO_URL = "https://github.com/huynhduc0/itmo-vkr-cefr.git"  # ← update if forked
REPO_DIR = "/kaggle/working/itmo-vkr-cefr"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Hugging Face token — read from Kaggle Secrets (needed for LLaMA-3.2 / Exp 4)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    import huggingface_hub
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print(f"⚠️  HF_TOKEN not available ({e}). Exp 4 (LLaMA) will be skipped.")

## 2. Configuration

In [ ]:
# ── User-configurable ────────────────────────────────────────────────────────

LANGUAGE = "en"       # en | ru | it | es | de | fr
TASK     = "sentence" # sentence | essay

EXPS_CPU         = [0, 1, 7, 9]            # no GPU needed
EXPS_TRANSFORMER = [2, 3, 8]               # XLM-R: CE, CORAL, zero-shot
EXPS_LLM         = [4] if HF_TOKEN else [] # LLaMA+LoRA — skipped without HF_TOKEN

# H100 / large-GPU overrides
BATCH_SIZE_TRANSFORMER = 64   # default 16 in config
BATCH_SIZE_LLM         = 8
NUM_EPOCHS             = 5
NUM_EPOCHS_LLM         = 3
USE_BF16               = True # H100 natively supports bf16

# ────────────────────────────────────────────────────────────────────────────
print(f"Language : {LANGUAGE}")
print(f"Task     : {TASK}")
print(f"CPU exps : {EXPS_CPU}")
print(f"GPU exps : {EXPS_TRANSFORMER}")
print(f"LLM exps : {EXPS_LLM}")

In [ ]:
from src import config as cfg

cfg.TRANSFORMER_CONFIG["batch_size"] = BATCH_SIZE_TRANSFORMER
cfg.TRANSFORMER_CONFIG["num_epochs"] = NUM_EPOCHS
cfg.LLM_CONFIG["batch_size"]         = BATCH_SIZE_LLM
cfg.LLM_CONFIG["num_epochs"]         = NUM_EPOCHS_LLM

if USE_BF16:
    os.environ["ACCELERATE_MIXED_PRECISION"]      = "bf16"
    os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
    print("✓ bf16 enabled via ACCELERATE_MIXED_PRECISION")

print(f"Transformer: batch={cfg.TRANSFORMER_CONFIG['batch_size']}, "
      f"epochs={cfg.TRANSFORMER_CONFIG['num_epochs']}")
print(f"LLM        : batch={cfg.LLM_CONFIG['batch_size']}, "
      f"epochs={cfg.LLM_CONFIG['num_epochs']}")

## 3. Data Preparation

`prepare_data.py` always creates **both** tracks (sentence + essay) in one run.

In [ ]:
import subprocess

DATA_DIR = "/kaggle/working/data"

# Note: no --task flag — prepare_data.py always writes both sentence/ and essay/
cmd = [
    sys.executable, "-m", "src.prepare_data",
    "--language", LANGUAGE,
    "--output", DATA_DIR,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("prepare_data failed")

# Show what was created
for track in ["sentence", "essay"]:
    for split in ["train", "dev", "test"]:
        path = os.path.join(DATA_DIR, track, f"{split}.jsonl")
        if os.path.exists(path):
            n = sum(1 for _ in open(path, encoding="utf-8"))
            print(f"  {track}/{split:5s}: {n:6,} examples")

In [ ]:
from src.run_experiments import _load_splits_from_jsonl
from src.data_utils import set_seed

set_seed(42)
(train_texts, train_labels), (val_texts, val_labels), (test_texts, test_labels) = \
    _load_splits_from_jsonl(DATA_DIR, TASK)

print(f"Track : {TASK}")
print(f"Train : {len(train_texts):,}")
print(f"Val   : {len(val_texts):,}")
print(f"Test  : {len(test_texts):,}")

from collections import Counter
from src.config import ID2LABEL
label_dist = Counter(ID2LABEL[l] for l in train_labels)
print("Train label distribution:", dict(sorted(label_dist.items())))

## 4. CPU Baseline Experiments (Exp 0, 1, 7, 9)

In [ ]:
from src.run_experiments import run_exp0, run_exp1, run_exp7, run_exp9
import time

all_results = []

if 0 in EXPS_CPU:
    print("▶ Exp 0 – Majority baseline")
    r = run_exp0(train_labels, test_labels, len(test_texts), track=TASK)
    all_results.append(r)
    print(f"   QWK={r.qwk:.4f}  Acc={r.accuracy:.4f}  [{r.note}]")

if 1 in EXPS_CPU:
    print("▶ Exp 1 – TF-IDF + Logistic Regression")
    t0 = time.time()
    r = run_exp1(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"   QWK={r.qwk:.4f}±{r.qwk_ci:.3f}  F1={r.macro_f1:.4f}±{r.macro_f1_ci:.3f}  "
          f"Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

if 7 in EXPS_CPU:
    print("▶ Exp 7 – TF-IDF + LinearSVC")
    t0 = time.time()
    r = run_exp7(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"   QWK={r.qwk:.4f}  F1={r.macro_f1:.4f}  Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

if 9 in EXPS_CPU:
    print("▶ Exp 9 – Word-only TF-IDF + LR")
    t0 = time.time()
    r = run_exp9(train_texts, train_labels, test_texts, test_labels, track=TASK)
    all_results.append(r)
    print(f"   QWK={r.qwk:.4f}  F1={r.macro_f1:.4f}  Acc={r.accuracy:.4f}  ({time.time()-t0:.1f}s)")

print("\n✓ CPU experiments done")

## 5. Transformer Experiments (Exp 2, 3, 8) — GPU

In [ ]:
if 2 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp2
    print("▶ Exp 2 – XLM-R fine-tuned (cross-entropy)")
    t0 = time.time()
    r2 = run_exp2(
        train_texts, train_labels,
        val_texts,   val_labels,
        test_texts,  test_labels,
        track=TASK, language=LANGUAGE,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE_TRANSFORMER, seed=42,
    )
    print(f"   QWK={r2.qwk:.4f}±{r2.qwk_ci:.3f}  F1={r2.macro_f1:.4f}±{r2.macro_f1_ci:.3f}  "
          f"Acc={r2.accuracy:.4f}  MAE={r2.mae:.4f}  ({(time.time()-t0)/60:.1f} min)")
    all_results.append(r2)
else:
    r2 = None
    print("Exp 2 skipped")

In [ ]:
if 3 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp3
    print("▶ Exp 3 – CORAL ordinal regression (fixed threshold init)")
    t0 = time.time()
    r3 = run_exp3(
        train_texts, train_labels,
        val_texts,   val_labels,
        test_texts,  test_labels,
        track=TASK, language=LANGUAGE,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE_TRANSFORMER, seed=42,
    )
    print(f"   QWK={r3.qwk:.4f}±{r3.qwk_ci:.3f}  F1={r3.macro_f1:.4f}±{r3.macro_f1_ci:.3f}  "
          f"Acc={r3.accuracy:.4f}  MAE={r3.mae:.4f}  ({(time.time()-t0)/60:.1f} min)")
    if r2:
        print(f"   CORAL vs XLM-R CE: ΔQWK = {r3.qwk - r2.qwk:+.4f}")
    all_results.append(r3)
else:
    r3 = None
    print("Exp 3 skipped")

In [ ]:
if 8 in EXPS_TRANSFORMER:
    from src.run_experiments import run_exp8
    print(f"▶ Exp 8 – Zero-shot XLM-R (train=en → eval={LANGUAGE})")
    t0 = time.time()
    r8 = run_exp8(
        train_texts=train_texts, train_labels=train_labels,
        test_texts=test_texts,   test_labels=test_labels,
        track=TASK, language=LANGUAGE, mode="zero_shot", seed=42,
    )
    print(f"   QWK={r8.qwk:.4f}±{r8.qwk_ci:.3f}  F1={r8.macro_f1:.4f}±{r8.macro_f1_ci:.3f}  "
          f"Acc={r8.accuracy:.4f}  ({(time.time()-t0)/60:.1f} min)")
    if r2:
        print(f"   Zero-shot vs fine-tuned: ΔQWK = {r8.qwk - r2.qwk:+.4f} (transfer cost)")
    all_results.append(r8)
else:
    r8 = None
    print("Exp 8 skipped")

## 6. LLM Experiment (Exp 4) — LLaMA-3.2 + LoRA

Requires `HF_TOKEN` set above and the LLaMA-3.2 license accepted on HF Hub.  
Returns **two** results: `raw` (regex, shows hallucination) and `constrained` (log-prob scoring, fair evaluation).

In [ ]:
if 4 in EXPS_LLM:
    from src.run_experiments import run_exp4
    print("▶ Exp 4 – LLaMA-3.2-3B + LoRA (QLoRA 4-bit)")
    t0 = time.time()
    r4_raw, r4_constrained = run_exp4(
        train_texts, train_labels,
        val_texts,   val_labels,
        test_texts,  test_labels,
        track=TASK, language=LANGUAGE, seed=42,
    )
    elapsed = (time.time() - t0) / 60
    print(f"   RAW         QWK={r4_raw.qwk:.4f}  F1={r4_raw.macro_f1:.4f}  [{r4_raw.note}]")
    print(f"   CONSTRAINED QWK={r4_constrained.qwk:.4f}±{r4_constrained.qwk_ci:.3f}  "
          f"F1={r4_constrained.macro_f1:.4f}  ({elapsed:.1f} min)")
    all_results.extend([r4_raw, r4_constrained])
else:
    print("Exp 4 skipped (no HF_TOKEN or not selected)")

## 7. Results Summary

In [ ]:
from src.run_experiments import print_comparison_table
print(f"\nResults — language={LANGUAGE}, task={TASK}")
print_comparison_table(all_results)

In [ ]:
import json
from src.run_experiments import save_results_to_files

OUT_DIR = f"/kaggle/working/results/{TASK}/{LANGUAGE}"
save_results_to_files(all_results, OUT_DIR)

with open(os.path.join(OUT_DIR, "results.json")) as f:
    records = json.load(f)

print(f"\n{'Experiment':<48} {'QWK':>10} {'±CI':>7} {'F1':>8} {'Acc':>7}")
print("-" * 84)
for rec in records:
    ci = f"±{rec['qwk_ci']:.3f}" if rec.get("qwk_ci") else ""
    print(f"{rec['name']:<48} {rec['qwk']:>10.4f} {ci:>7} "
          f"{rec['macro_f1']:>8.4f} {rec['accuracy']:>7.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

names  = [r["name"].replace(" – ", "\n") for r in records]
qwks   = [r["qwk"] for r in records]
cis    = [r.get("qwk_ci", 0.0) for r in records]

COLOR_MAP = {
    lambda n: "LLM" in n or "LoRA" in n:                  "#e07b39",
    lambda n: "CORAL" in n or "Ordinal" in n:              "#5b9bd5",
    lambda n: "Transformer" in n or "XLM-R" in n:         "#4caf50",
}
def get_color(name):
    for pred, color in COLOR_MAP.items():
        if pred(name):
            return color
    return "#9e9e9e"
colors = [get_color(r["name"]) for r in records]

fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.6)))
y = np.arange(len(names))
ax.barh(y, qwks, xerr=cis, align="center", height=0.6,
        color=colors, capsize=4, error_kw={"elinewidth": 1.5})
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Quadratic Weighted Kappa (QWK)")
ax.set_title(f"CEFR — {LANGUAGE.upper()} / {TASK}  (QWK ± 95% CI)", fontsize=12)
ax.set_xlim(0, 1.05)
ax.invert_yaxis()

for i, (v, ci) in enumerate(zip(qwks, cis)):
    label = f"{v:.3f}" + (f"±{ci:.3f}" if ci else "")
    ax.text(min(v + 0.01, 1.0), i, label, va="center", fontsize=8)

ax.legend(handles=[
    mpatches.Patch(color="#9e9e9e", label="Baseline (TF-IDF)"),
    mpatches.Patch(color="#4caf50", label="Transformer (XLM-R)"),
    mpatches.Patch(color="#5b9bd5", label="Ordinal (CORAL)"),
    mpatches.Patch(color="#e07b39", label="LLM (LLaMA+LoRA)"),
], loc="lower right", fontsize=9)

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "qwk_bar.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {plot_path}")

## 8. (Optional) Multilingual Sweep — All 6 Languages

Uncomment to run Exp 1 + Exp 2 + Exp 8 for every language.  
Note: `prepare_data.py` takes no `--task` flag — it always writes both tracks.

In [ ]:
# UNCOMMENT to sweep all languages
# from src.config import SUPPORTED_LANGUAGES
# from src.run_experiments import (
#     run_exp0, run_exp1, run_exp2, run_exp8,
#     _load_splits_from_jsonl, save_results_to_files, print_comparison_table,
# )
# from src.data_utils import set_seed
# import subprocess, time
#
# ALL_RESULTS = {}
# for lang in SUPPORTED_LANGUAGES:
#     print(f"\n{'='*55}  {lang.upper()}  {'='*55}")
#     lang_data = f"/kaggle/working/data_{lang}"
#     # prepare_data writes both sentence/ and essay/ — no --task flag
#     subprocess.run(
#         [sys.executable, "-m", "src.prepare_data",
#          "--language", lang, "--output", lang_data],
#         cwd=REPO_DIR, check=True,
#     )
#     (tr_t, tr_l), (vl_t, vl_l), (te_t, te_l) = _load_splits_from_jsonl(lang_data, TASK)
#     set_seed(42)
#     lang_results = [
#         run_exp0(tr_l, te_l, len(te_t), track=TASK),
#         run_exp1(tr_t, tr_l, te_t, te_l, track=TASK),
#         run_exp2(tr_t, tr_l, vl_t, vl_l, te_t, te_l, track=TASK, language=lang, seed=42),
#     ]
#     if lang != "en":
#         lang_results.append(
#             run_exp8(tr_t, tr_l, te_t, te_l, track=TASK, language=lang,
#                      mode="zero_shot", seed=42)
#         )
#     save_results_to_files(lang_results, f"/kaggle/working/results/{TASK}/{lang}")
#     ALL_RESULTS[lang] = lang_results
#     print_comparison_table(lang_results)
# print("\n✓ All languages done")
print("Multilingual sweep cell — uncomment to run")

In [ ]:
print("=" * 60)
print(f"Language : {LANGUAGE}")
print(f"Task     : {TASK}")
print(f"Results  : {OUT_DIR}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if all_results:
    best = max(all_results, key=lambda r: r.qwk)
    print(f"Best QWK : {best.qwk:.4f}±{best.qwk_ci:.3f}  ({best.name})")
print("=" * 60)